# Native REINFORCE Quickstart

This notebook shows the shortest path to training a native `crosslearn.REINFORCE` agent on a standard Gymnasium task. The code below uses `CartPole-v1`, but the same flow works for `LunarLander-v3` by changing the environment id and the solved-threshold callback.
            

In [ ]:
# Uncomment in a fresh environment:
# %pip install crosslearn
            

In [ ]:
import gymnasium as gym

from crosslearn import REINFORCE, make_vec_env
from crosslearn.callbacks import EpisodeSolvedCallback
            

## Choose the environment

The package vectorizes environments internally, so the native agent sees batched observations even when you start from a single Gymnasium id.
            

In [ ]:
ENV_ID = 'CartPole-v1'
N_ENVS = 4

vec_env = make_vec_env(ENV_ID, n_envs=N_ENVS)
print('Vector env type:', type(vec_env).__name__)
print('Num envs:', vec_env.num_envs)
print('Single observation space:', vec_env.single_observation_space)
print('Single action space:', vec_env.single_action_space)
            

## Train with the native defaults

The default configuration is enough for a first run. In practice, the native REINFORCE settings most worth tuning are:

- `n_steps`
- `gamma`
- `learning_rate`
- `normalize_returns`
- `entropy_coeff`
- `max_grad_norm`
- `policy_kwargs`
- `features_extractor_class` and `features_extractor_kwargs`
- optimizer and scheduler classes
- `seed`, `device`, and optional logger integration

For SB3 algorithms, refer to the SB3 documentation for PPO-specific rollout and optimization settings.
            

In [ ]:
reward_threshold = 475.0 if ENV_ID == 'CartPole-v1' else 200.0

agent = REINFORCE(vec_env, seed=42, verbose=1)
agent.learn(
    total_timesteps=100_000,
    callbacks=[EpisodeSolvedCallback(reward_threshold=reward_threshold, n_episodes=100)],
)
            

## Run a deterministic evaluation episode

Use `predict(..., deterministic=True)` for a quick post-training rollout.
            

In [ ]:
eval_env = gym.make(ENV_ID, render_mode='rgb_array')
obs, info = eval_env.reset(seed=42)
terminated = False
truncated = False
episode_return = 0.0

while not (terminated or truncated):
    action = int(agent.predict(obs, deterministic=True))
    obs, reward, terminated, truncated, info = eval_env.step(action)
    episode_return += reward

print(f'Evaluation return on {ENV_ID}: {episode_return:.2f}')
eval_env.close()
            